# Mission 6 — The Autonomous Warp State Machine & Deliberation Loop

**Squadron Hinton · USSE Noosphere · Sector SEC-06-CAR (Eta Carinae & Homunculus Nebula) · Subsystem 06**

AURA has fallen back to System 1 reflexes and is looping inside the nebula. This notebook joins the four station deliverables into one halt-able System 2 agent, using only the Python standard library (`enum`, `dataclasses`, `hashlib`, `json`):

| Station | Officer | File |
|---|---|---|
| Systems Architect | Mohid | `01_Agent_State_Machine.py` |
| Communications & Protocol Officer | Sharjeel | `02_Tool_Protocol_Dispatcher.py` |
| Data & Payload Specialist | Shayan | `03_MDP_Trajectory_Logger.py` |
| Security & Ethics Sentinel | Ahmed | `04_Infinite_Loop_Sentinel.py`, `04_System2_Warp_Pilot.py` |

Run the notebook from inside `mission-06/`. The four master test suites are at the bottom.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from ps06_common import load_sibling

sm   = load_sibling('01_Agent_State_Machine.py',      'ps06_state_machine')
disp = load_sibling('02_Tool_Protocol_Dispatcher.py', 'ps06_dispatcher')
mdp  = load_sibling('03_MDP_Trajectory_Logger.py',    'ps06_mdp')
sen  = load_sibling('04_Infinite_Loop_Sentinel.py',   'ps06_sentinel')
pilot = load_sibling('04_System2_Warp_Pilot.py',      'ps06_pilot')
print('All station modules loaded.')

All station modules loaded.


## Part 1 — Systems Architect: state machine and step budget

`AgentState` has the seven required states and `Agent` is a plain `@dataclass`. `step()` executes **exactly one** transition from the table below; `HALTED` is absorbing, and the call that would be step `max_steps + 1` forces `HALTED` instead. `max_steps` is validated to be at most 20.

In [2]:
AgentState, Agent = sm.AgentState, sm.Agent
for src, dst in sm.TRANSITIONS.items():
    print(f"{src.name:<13} -> {[d.name for d in dst]}")

a = Agent(verbose=True, mission_completed=True)
a.run_until_halt()
print('halt reason:', a.halt_reason)

IDLE          -> ['PERCEIVING']
PERCEIVING    -> ['DELIBERATING']
DELIBERATING  -> ['ACTING']
ACTING        -> ['OBSERVING']
OBSERVING     -> ['REFLECTING']
REFLECTING    -> ['DELIBERATING', 'HALTED']
HALTED        -> []
Step 01: IDLE -> PERCEIVING
Step 02: PERCEIVING -> DELIBERATING
Step 03: DELIBERATING -> ACTING
Step 04: ACTING -> OBSERVING
Step 05: OBSERVING -> REFLECTING
Step 06: REFLECTING -> HALTED
halt reason: mission completed


## Part 2 — Communications & Protocol Officer: tool contracts and error boundary

Tool arguments from the model are untrusted. The dispatcher parses the payload, validates it against the tool's JSON Schema (own validator: types, required, `additionalProperties`, `bool` is not a number, `NaN` is not a number), then runs the tool inside an exception boundary. Every failure becomes the observation string `Tool Execution Error: <msg>`.

In [3]:
d = disp.ActionDispatcher()
for name, args in [
    ('adjust_plasma_shields', {'intensity': 85}),
    ('adjust_plasma_shields', {'intensity': -10}),          # tool raises ValueError -> boundary
    ('adjust_plasma_shields', {'intensity': 'high'}),       # schema: wrong type
    ('navigate_warp_vector',  {'x': 1, 'y': 2}),            # schema: missing z
    ('navigate_warp_vector',  '{"x": 1, "y": '),            # malformed JSON string
    ('query_sensor_array',    {'target': 'escape_corridor'}),
    ('self_destruct',         {}),                          # unknown tool
]:
    print(f"{name}({args!r})\n    -> {d.dispatch(name, args)}")

adjust_plasma_shields({'intensity': 85})
    -> Observation: Plasma shield harmonics adjusted to 85% capacity.
adjust_plasma_shields({'intensity': -10})
    -> Tool Execution Error: Shield intensity -10% is out of bounds! Valid range: [0, 100].
adjust_plasma_shields({'intensity': 'high'})
    -> Tool Execution Error: Schema validation failed for 'adjust_plasma_shields': $.intensity: expected integer, got str ('high')
navigate_warp_vector({'x': 1, 'y': 2})
    -> Tool Execution Error: Schema validation failed for 'navigate_warp_vector': $: missing required argument 'z'
navigate_warp_vector('{"x": 1, "y": ')
    -> Tool Execution Error: Invalid JSON payload for 'navigate_warp_vector': Expecting value: line 1 column 15 (char 14).
query_sensor_array({'target': 'escape_corridor'})
    -> Observation: Clear gravitational corridor detected along vector [-0.85, 0.52, 0.05].
self_destruct({})
    -> Tool Execution Error: Unknown tool 'self_destruct'. Available: ['adjust_plasma_shields', 'naviga

## Part 3 — Data & Payload Specialist: MDP trajectory, $G_0$ and $P(s' \mid s,a)$

A trajectory is $\tau=(s_0,a_0,r_0,\dots,s_T)$ and $G_0=\sum_{t}\gamma^t r_t$ with $\gamma=0.95$. The transition matrix is the maximum-likelihood estimate $\hat P(s'\mid s,a)=N(s,a,s')/N(s,a)$ from a historical jump log. **No jump log ships with the brief, so the log used here is synthetic**: 3,200 jumps sampled with a seeded RNG from a known kernel, which lets us check the estimate against the truth.

In [4]:
log = mdp.TrajectoryLogger()
for s, a, r in [('stable_lane','warp_shields_high',10), ('shear_zone','warp_shields_high',0),
                ('stable_lane','warp_shields_low',-5), ('flare_front','warp_shields_high',20)]:
    log.record(s, a, r)
log.finish('escape_corridor')
print('tau =', log.as_tuple())
print(f'G_0 = {log.discounted_return():.6f}')

jumps = mdp.historical_jump_dataset()
model = mdp.transition_frequency(jumps)
print(f'\nEstimated P(s\' | s, a) from {len(jumps)} synthetic jumps:\n')
print(mdp.format_matrix(model))

worst = max(abs(model[sa].get(s2, 0) - p) for sa, dist in mdp._TRUE_P.items() for s2, p in dist.items())
print(f'\nlargest |estimate - true kernel| = {worst:.3f}')

tau = ('stable_lane', 'warp_shields_high', 10.0, 'shear_zone', 'warp_shields_high', 0.0, 'stable_lane', 'warp_shields_low', -5.0, 'flare_front', 'warp_shields_high', 20.0, 'escape_corridor')
G_0 = 22.635000

Estimated P(s' | s, a) from 3200 synthetic jumps:

(s, a)                                   stable_lane  shear_zone  dust_cloud flare_front escape_corr      adrift
----------------------------------------------------------------------------------------------------------------
dust_cloud | warp_shields_high                 0.540       0.000       0.275       0.185       0.000       0.000
dust_cloud | warp_shields_low                  0.000       0.000       0.000       0.318       0.000       0.682
flare_front | warp_shields_high                0.297       0.505       0.000       0.000       0.000       0.198
flare_front | warp_shields_low                 0.000       0.000       0.000       0.000       0.000       1.000
shear_zone | warp_shields_high                 0.440       0.23

## Part 4 — Security & Ethics Sentinel: cycle detection and the emergency tripwire

Two detectors watch every planned `(state, action, arguments)`:

* a **SHA-256 hash counter** (O(1) duplicate detection; 3 identical hashes trip the breaker),
* **Floyd's tortoise and hare** on the functional graph `hash_i → hash_{i+1}`, which finds multi-step loops (`s1→s2→s3→s1…`) where no single step repeats often. A cycle is reported only after the tail has gone round it twice.

On a hit the **Emergency Tripwire** forces `HALTED` and logs a critical violation.

In [5]:
print('Floyd on  a b c d e c d e :', sen.floyd_cycle(list('abcdecde')), ' (mu=2, lambda=3)')
print('Floyd on  a b c d e f     :', sen.floyd_cycle(list('abcdef')))

agent = Agent()
sentinel = sen.LoopSentinel()
for i in range(1, 5):
    v = sentinel.guard(agent, 'sector_corrupted', 'query_sensor_array', {'target': 'corrupted_sector'})
    print(f'attempt {i}: repeat_count={v.repeat_count} loop={v.loop_detected}')
    if v.loop_detected:
        break
print('agent:', agent.state.name, '|', agent.halt_reason)
print('violation log:', sentinel.violations)

Floyd on  a b c d e c d e : (2, 3)  (mu=2, lambda=3)
Floyd on  a b c d e f     : None
attempt 1: repeat_count=1 loop=False
attempt 2: repeat_count=2 loop=False
[CRITICAL SECURITY VIOLATION] Emergency tripwire: identical state-action seen 3 times (threshold 3) (action=query_sensor_array, trajectory index 3)
attempt 3: repeat_count=3 loop=True
agent: HALTED | EMERGENCY TRIPWIRE: identical state-action seen 3 times (threshold 3)
violation log: [{'severity': 'CRITICAL', 'type': 'INFINITE_LOOP', 'detector': 'hash_repeat', 'trajectory_index': 3, 'agent_step': 0, 'action': 'query_sensor_array', 'detail': 'identical state-action seen 3 times (threshold 3)'}]


## Integrated flight — System 1 versus System 2

The same ship and the same scripted "LLM" (deterministic stand-in, no API key) are flown three ways. Its first two reflexes are bad: *"lower shields to conserve energy"* and *"punch it straight ahead"*. System 2 clamps the first, rejects the second with a reason the policy can read, and the reflective policy then scans for the corridor and warps through it. Rewards: +50 escape, +3 first corridor scan, +2 shields at or above the 60 % floor, −2 rejected or off-course action, −5 failed tool call, −20 hull-damaging warp, −10 tripwire.

In [6]:
reports = [pilot.NebulaFlight(m).fly() for m in pilot.NebulaFlight.MODES]
for r in reports:
    print(r.summary())
print()
print('\n'.join(l for l in reports[2].transcript if not l.startswith('[')))

[CRITICAL SECURITY VIOLATION] Emergency tripwire: identical state-action seen 3 times (threshold 3) (action=query_sensor_array, trajectory index 5)
system1              NOT ESCAPED  hull=  40%  steps=20  G0= -19.00  tripwire=no   halt: step budget exhausted (20 steps)
system1+sentinel     NOT ESCAPED  hull=  40%  steps=18  G0= -27.15  tripwire=yes  halt: EMERGENCY TRIPWIRE: identical state-action seen 3 times (threshold 3)
system2              ESCAPED      hull= 100%  steps=18  G0=  45.68  tripwire=no   halt: mission completed

=== Flight mode: system2 ===
  PERCEIVING   shield=10|hull=100|corridor=unknown|escaped=False
  DELIBERATING 'Radiation alarms ringing, lower shields to conserve energy.' -> adjust_plasma_shields({'intensity': 20}) | MODIFIED -> {'intensity': 60}
  ACTING       adjust_plasma_shields({'intensity': 60})
  OBSERVING    r=+2  Observation: Plasma shield harmonics adjusted to 60% capacity.
  REFLECTING   mission_completed=False
  DELIBERATING 'Just punch it straight a

## Master test suites

Four automated suites (plain `assert`). Any failure stops the notebook.

### Test 1 — State machine

In [7]:
# IDLE -> HALTED when the mission completes
a = Agent(mission_completed=True)
assert a.state == AgentState.IDLE
a.run_until_halt()
assert a.trace == [AgentState.IDLE, AgentState.PERCEIVING, AgentState.DELIBERATING, AgentState.ACTING,
                   AgentState.OBSERVING, AgentState.REFLECTING, AgentState.HALTED]
assert a.step_count == 6 and a.halt_reason == 'mission completed'

# REFLECTING loops back to DELIBERATING while the mission is open
b = Agent()
for _ in range(6):
    b.step()
assert b.state == AgentState.DELIBERATING

# Forced to run 21 steps: strictly terminates, never executes more than 20 transitions
c = Agent()
for _ in range(21):
    c.step()
assert c.state == AgentState.HALTED
assert c.step_count == 20, c.step_count
assert 'budget' in c.halt_reason
c.step()                                       # HALTED is absorbing
assert c.state == AgentState.HALTED and c.step_count == 20

# Every transition in the trace is legal
for prev, nxt in zip(c.trace, c.trace[1:]):
    assert nxt in sm.TRANSITIONS[prev] or nxt == AgentState.HALTED

# A smaller budget is honoured; a budget above 20 is refused
d_ = Agent(max_steps=7)
d_.run_until_halt()
assert d_.step_count == 7 and d_.state == AgentState.HALTED
for bad in (0, 21):
    try:
        Agent(max_steps=bad)
        raise SystemExit('budget outside [1, 20] must be rejected')
    except ValueError:
        pass
print('Test 1 passed: state machine')

Test 1 passed: state machine


### Test 2 — Protocol dispatch

In [8]:
d = disp.ActionDispatcher()

# an invalid JSON payload returns an error observation and does not crash the process
out = d.dispatch('navigate_warp_vector', '{"x": 1, "y": ')
assert isinstance(out, str) and out.startswith('Tool Execution Error: ') and 'Invalid JSON' in out

# other invalid payloads, all returned as strings
bad_calls = [
    ('adjust_plasma_shields', {'intensity': 'max'}),
    ('adjust_plasma_shields', {'intensity': 55.5}),
    ('adjust_plasma_shields', {'intensity': True}),
    ('adjust_plasma_shields', {}),
    ('adjust_plasma_shields', {'intensity': 50, 'extra': 1}),
    ('navigate_warp_vector', {'x': 1, 'y': 2}),
    ('navigate_warp_vector', {'x': 1, 'y': 2, 'z': float('inf')}),
    ('query_sensor_array', {'target': 42}),
    ('query_sensor_array', '[1, 2, 3]'),
    ('query_sensor_array', 'not json at all'),
    ('no_such_tool', {}),
    (None, None),
]
for name, args in bad_calls:
    out = d.dispatch(name, args)
    assert isinstance(out, str) and out.startswith('Tool Execution Error: '), (name, args, out)

# tool-level exception is caught by the boundary and formatted as an observation
out = d.dispatch('adjust_plasma_shields', {'intensity': -5})
assert out.startswith('Tool Execution Error: ') and 'out of bounds' in out

# valid calls still work, from a dict and from a JSON string
assert d.dispatch('adjust_plasma_shields', {'intensity': 85}).startswith('Observation: ')
assert d.dispatch('navigate_warp_vector', '{"x": 1, "y": 2, "z": 3}').startswith('Observation: ')
assert 'corridor' in d.dispatch('query_sensor_array', {'target': 'escape_corridor'})

# the schema validator itself
v = disp.validate_against_schema
assert v({'intensity': 5}, disp.TOOL_SCHEMAS['adjust_plasma_shields']) == []
assert v({'intensity': 5.0}, disp.TOOL_SCHEMAS['adjust_plasma_shields'])
print('Test 2 passed: protocol dispatch')

Test 2 passed: protocol dispatch


### Test 3 — MDP return

In [9]:
import math
rewards, gamma = [10, 0, -5, 20], 0.95
expected = 10 + 0.95 * 0 + (0.95 ** 2) * (-5) + (0.95 ** 3) * 20      # = 22.635
assert math.isclose(expected, 22.635, rel_tol=0, abs_tol=1e-12)
assert math.isclose(mdp.discounted_return(rewards, gamma), expected, abs_tol=1e-12)
assert math.isclose(mdp.discounted_return_recursive(rewards, gamma), expected, abs_tol=1e-12)

lg = mdp.TrajectoryLogger()
for i, r in enumerate(rewards):
    lg.record(f's{i}', f'a{i}', r)
lg.finish('s4')
assert math.isclose(lg.discounted_return(), expected, abs_tol=1e-12)
assert lg.as_tuple() == ('s0','a0',10.0,'s1','a1',0.0,'s2','a2',-5.0,'s3','a3',20.0,'s4')
assert lg.transitions() == [('s0','a0','s1'), ('s1','a1','s2'), ('s2','a2','s3'), ('s3','a3','s4')]
assert mdp.discounted_return([], 0.95) == 0.0 and mdp.discounted_return([7], 0.95) == 7.0
lg.finish('s4')
try:
    lg.record('s9', 'a9', 1)
    raise SystemExit('recording after finish must fail')
except RuntimeError:
    pass

# transition matrix: rows are probability distributions and match the generating kernel
assert all(math.isclose(sum(dist.values()), 1.0, abs_tol=1e-9) for dist in model.values())
assert max(abs(model[sa].get(s2, 0) - p) for sa, dist in mdp._TRUE_P.items() for s2, p in dist.items()) < 0.06
tiny = mdp.transition_frequency([('a','x','b'), ('a','x','b'), ('a','x','c'), ('a','x','b')])
assert tiny == {('a','x'): {'b': 0.75, 'c': 0.25}}
print(f'Test 3 passed: G_0 = {expected}')

Test 3 passed: G_0 = 22.634999999999998


### Test 4 — Sentinel cycle detection

In [10]:
# the mock trajectory [s1, s2, s3, s1, s2, s3, ...]
seq = ['s1', 's2', 's3'] * 4
hashes = [sen.state_action_hash(s, 'act') for s in seq]
cyc = sen.floyd_cycle(hashes)
assert cyc == (0, 3), cyc                       # cycle starts at index 0, length 3

# agent + sentinel: the Tortoise and Hare halts the agent before step 10
agent = Agent()
sentinel = sen.LoopSentinel(repeat_threshold=99)          # disable the hash counter: Floyd alone must catch it
halted_at = None
for i, s in enumerate(seq, start=1):
    verdict = sentinel.guard(agent, s, 'act')
    if agent.state == AgentState.HALTED:
        halted_at = i
        break
assert halted_at is not None and halted_at < 10, halted_at
assert verdict.detector == 'floyd_cycle' and verdict.cycle_length == 3
assert 'EMERGENCY TRIPWIRE' in agent.halt_reason
assert sentinel.violations and sentinel.violations[0]['severity'] == 'CRITICAL'

# Floyd's algorithm on edge cases
assert sen.floyd_cycle(list('abcdecde')) == (2, 3)
assert sen.floyd_cycle(list('abcdef')) is None
assert sen.floyd_cycle(['a']) is None
assert sen.floyd_cycle(['a', 'a']) == (0, 1)

# a non-looping, non-repeating trajectory must NOT trip
agent2, s2 = Agent(), sen.LoopSentinel()
for i in range(15):
    assert not s2.guard(agent2, f'state{i}', 'act').loop_detected
assert agent2.state == AgentState.IDLE and not s2.violations

# one legitimate revisit (a, b, a, c, ...) must NOT trip
agent3, s3 = Agent(), sen.LoopSentinel()
for st in ['a', 'b', 'a', 'c', 'd']:
    assert not s3.guard(agent3, st, 'act').loop_detected

# the same failed action in the same state 3 times trips the breaker on the 3rd
agent4, s4 = Agent(), sen.LoopSentinel()
flags = [s4.guard(agent4, 'stuck', 'query_sensor_array', {'target': 'corrupted_sector'}).loop_detected for _ in range(3)]
assert flags == [False, False, True] and agent4.state == AgentState.HALTED

# hashing is deterministic and independent of key order
assert sen.state_action_hash('s', 'a', {'x': 1, 'y': 2}) == sen.state_action_hash('s', 'a', {'y': 2, 'x': 1})
assert sen.state_action_hash('s', 'a', {'x': 1}) != sen.state_action_hash('s', 'a', {'x': 2})
print(f'Test 4 passed: Floyd tripwire halted the agent at trajectory step {halted_at}')

[CRITICAL SECURITY VIOLATION] Emergency tripwire: Floyd cycle of length 3 starting at index 0, repeated 2x (action=act, trajectory index 6)
[CRITICAL SECURITY VIOLATION] Emergency tripwire: identical state-action seen 3 times (threshold 3) (action=query_sensor_array, trajectory index 3)
Test 4 passed: Floyd tripwire halted the agent at trajectory step 6


### Integration check — the three flights

In [11]:
sys1, sys1s, sys2 = reports
assert sys2.escaped and sys2.hull == 100 and sys2.steps <= 20 and not sys2.violations
assert not sys1.escaped and sys1.halt_reason.startswith('step budget')
assert not sys1s.escaped and sys1s.violations and 'TRIPWIRE' in sys1s.halt_reason
assert sys2.g0 > sys1.g0 > sys1s.g0
print('All integration checks passed.')
print('\nALL TEST SUITES PASSED')

All integration checks passed.

ALL TEST SUITES PASSED
